# Lab 08 — Multi-Agent Swarms

## Objective

Run independent reviewers concurrently and route conflicts to a human.

## Introduction and concept

ThreadPoolExecutor runs all reviewers, and a barrier requires concurrent arrival before checks proceed. The merge surfaces disagreements.

## Real-world use case

Review a draft question for coverage, clarity and difficulty before publication.

## Technology stack and dependencies

Python concurrent.futures and threading. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Coordinator → parallel reviewer fan-out → merge → publish, revise or human review


## Step 1 — Implement the components

ThreadPoolExecutor runs all reviewers, and a barrier requires concurrent arrival before checks proceed. The merge surfaces disagreements.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier

def coverage(question):
    return "join" in question.lower()

def clarity(question):
    return question.endswith("?") and len(question.split()) >= 5

def difficulty(question):
    return "explain" in question.lower()

AGENTS = {"coverage": coverage, "clarity": clarity, "difficulty": difficulty}

def coordinate(question):
    start = Barrier(len(AGENTS))
    def run_check(name):
        start.wait(timeout=5)
        return {"agent": name, "approved": AGENTS[name](question)}
    with ThreadPoolExecutor(max_workers=len(AGENTS)) as pool:
        results = list(pool.map(run_check, AGENTS))
    votes = {result["approved"] for result in results}
    if len(votes) > 1:
        decision = "human_review"
    else:
        decision = "publish" if votes.pop() else "revise"
    return {"trace": ["fan_out", *results, "merge"], "decision": decision}

## Step 2 — Run sample inputs and inspect actual output

Review three questions that produce unanimous approval, disagreement and unanimous rejection. Look for the fan-out and merge trace.

In [2]:
def main():
    for question, expected in [
        ("Explain how an inner join works?", "publish"),
        ("Is an inner join required?", "human_review"),
        ("Index", "revise"),
    ]:
        result = coordinate(question)
        print(question, result)
        assert result["decision"] == expected
        assert len(result["trace"]) == len(AGENTS) + 2
    print("PASS: parallel rendezvous and disagreement handling")

main()

Explain how an inner join works? {'trace': ['fan_out', {'agent': 'coverage', 'approved': True}, {'agent': 'clarity', 'approved': True}, {'agent': 'difficulty', 'approved': True}, 'merge'], 'decision': 'publish'}
Is an inner join required? {'trace': ['fan_out', {'agent': 'coverage', 'approved': True}, {'agent': 'clarity', 'approved': True}, {'agent': 'difficulty', 'approved': False}, 'merge'], 'decision': 'human_review'}
Index {'trace': ['fan_out', {'agent': 'coverage', 'approved': False}, {'agent': 'clarity', 'approved': False}, {'agent': 'difficulty', 'approved': False}, 'merge'], 'decision': 'revise'}
PASS: parallel rendezvous and disagreement handling


## Testing and validation

Review three questions that produce unanimous approval, disagreement and unanimous rejection. Look for the fan-out and merge trace. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The reviewers use fixed rules over local data and share one process. A production swarm would need independent models and failure policies.

## How this relates to ExamPilot

Proposed extension. ExamPilot has specialist generation and validation stages wired sequentially in backend/app/graph/exam_graph.py. This lab runs actual local concurrency separately.

## Conclusion

Run independent reviewers concurrently and route conflicts to a human. Use the trace and assertions to explain which decisions the workflow makes.